In [27]:
import h5py
import traceback
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
import scipy.signal as sc
from scipy.optimize import curve_fit
from scipy.ndimage import filters 
import tifffile as tiff
from scipy.signal import find_peaks
import plotly.express as px
import os
import csv
import pandas as pd
from roipoly import MultiRoi
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import ast
from AnalasysFunction import BurstC,clean_and_parse,motorSp,plotFR,devideTr,plotVolCal,VolToCalIdx,CalSmooth,CorrWindow,ChooseSpk,CalInt,VolToCalIdx,CalAmp,calculate_firing_rate,ChooseCom,LongLIST,SingleSpk,linear_model,quadratic_model,exponential_model,MeanRes,lagOptimaizre
from spike_detection_Qixixn2 import complex_bursts_detection,refine_single_spikes,spike_height_calculation,detect_complex_spikes,refine_all_spikes,plot_trace_with_spikes_pdf,plot_trace_with_spikes_export,plot_trace_with_spikes_html
# Create dictionary
from TRY import LongLIST
from NewinternueronsAnalsys import analyze_block
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, linregress,ttest_ind
import pickle
from matplotlib.widgets import Slider, Button

In [28]:
def BS_change_diff(cal, vol, calax, volax,changP,motor,spike_idx_v,window = 3,Vsr = 500, Csr = 30,fr_bin_s=0.0333):
    motToRestVol = []
    motToRestCal = []
    motToRestFR  = []
    motToRestCalMean = []
    motToRestFRMean  = []
    motToRestFRMeanDiff  = []

    restToMotVol = []
    restToMotCal = []
    restToMotFR  = []
    restToMotCalMean = []
    restToMotFRMean  = []
    restToMotFRMeanDiff  = []
    spike_idx_v = np.asarray(spike_idx_v, dtype=int)

    for i in changP:
        startVolIdx = max(0, i - window*Vsr)
        endVolIdx = min(i + window*Vsr, len(vol) - 1)
        StartCalidx = VolToCalIdx(startVolIdx,volax,calax)
        endCalIdx = VolToCalIdx(endVolIdx,volax,calax)
        chCalIdx = VolToCalIdx(i,volax,calax)
        # spikes inside window
        sp = spike_idx_v[(spike_idx_v >= startVolIdx) & (spike_idx_v <= endVolIdx)]
        # convert to seconds relative to window start
        sp_t = (sp - startVolIdx) / float(Vsr)
        sp_Bef = spike_idx_v[(spike_idx_v >= startVolIdx) & (spike_idx_v <= i)]
        # convert to seconds relative to window start
        sp_t_bef = (sp_Bef - startVolIdx) / float(Vsr)
        sp_Aft = spike_idx_v[(spike_idx_v > i) & (spike_idx_v <= endVolIdx)]
        # convert to seconds relative to window start
        sp_t_aft = (sp_Aft - startVolIdx) / float(Vsr)

        win_dur_s = (endVolIdx - startVolIdx + 1) / float(Vsr)
        edges = np.arange(0.0, win_dur_s + fr_bin_s, fr_bin_s)
        if edges.size < 2:
            edges = np.array([0.0, fr_bin_s])

        counts, _ = np.histogram(sp_t, bins=edges)
        fr_win = counts / float(fr_bin_s)  # Hz per bin

        win_dur_s_bef = (i - startVolIdx + 1) / float(Vsr)
        edges = np.arange(0.0, win_dur_s_bef + fr_bin_s, fr_bin_s)
        if edges.size < 2:
            edges = np.array([0.0, fr_bin_s])

        counts_bef, _ = np.histogram(sp_t_bef, bins=edges)
        counts_aft, _ = np.histogram(sp_t_aft, bins=edges)
        fr_win_bef = counts_bef / float(fr_bin_s)  # Hz per bin
        fr_win_aft = counts_aft / float(fr_bin_s)  # Hz per bin

        # index in FR bins corresponding to the change point
        change_t_s = (i - startVolIdx) / float(Vsr)
        change_bin = int(np.clip(np.floor(change_t_s / fr_bin_s), 0, len(fr_win)))
        fr_bef = fr_win[:change_bin]
        fr_aft = fr_win[change_bin:]
        fr_mean_diff = (np.nanmean(fr_aft) - np.nanmean(fr_bef)) if (fr_bef.size and fr_aft.size) else np.nan
        if motor[i-10]==1:
            motToRestCal.append(cal[StartCalidx:endCalIdx+1])
            motToRestVol.append(cal[startVolIdx:endVolIdx+1])
            calBef = cal[StartCalidx:chCalIdx]
            calAf = cal[chCalIdx:endCalIdx]
            motToRestCalMean.append(np.mean(calAf)-np.mean(calBef))
            motToRestFR.append(fr_win)
            motToRestFRMean.append([fr_mean_diff])
            motToRestFRMeanDiff.append([np.mean(fr_win_aft) - np.mean(fr_win_bef)])
            
        if motor[i-10]==0:
            restToMotCal.append(cal[StartCalidx:endCalIdx+1])
            restToMotVol.append(cal[startVolIdx:endVolIdx+1])
            calBef = cal[StartCalidx:chCalIdx]
            calAf = cal[chCalIdx:endCalIdx]
            restToMotCalMean.append((np.mean(calAf)-np.mean(calBef)))
            restToMotFR.append(fr_win)
            restToMotFRMean.append([fr_mean_diff])
            restToMotFRMeanDiff.append([np.mean(fr_win_aft) - np.mean(fr_win_bef)])    
    result  = dict(motorToRest={'Vol': motToRestVol, 'Cal': motToRestCal,'CalMean':motToRestCalMean, 'FR':motToRestFR, 'FRMean':motToRestFRMean, 'FRMeanDiff':motToRestFRMeanDiff},
                  restToMotor={'Vol': restToMotVol, 'Cal': restToMotCal,'CalMean':restToMotCalMean, 'FR':restToMotFR, 'FRMean':restToMotFRMean, 'FRMeanDiff':restToMotFRMeanDiff})
    return result

In [29]:


def _stack_trim_to_minlen(traces):
    """Return (stack, min_len). stack shape = (n_events, min_len)."""
    traces = [np.asarray(t).flatten() for t in traces if t is not None and len(t) > 0]
    if len(traces) == 0:
        return None, 0
    min_len = min(len(t) for t in traces)
    stack = np.vstack([t[:min_len] for t in traces])
    return stack, min_len

def plot_all_gray_mean_black(ax, traces, fs=None, title=""):
    """
    traces: list of 1D arrays. Plots all (gray transparent) + mean (black).
    fs: if provided, x-axis in seconds; else samples.
    """
    stack, L = _stack_trim_to_minlen(traces)
    if stack is None:
        ax.set_title(title + " (no data)")
        return None

    if fs is None:
        x = np.arange(L)
        ax.set_xlabel("Samples")
    else:
        x = np.arange(L) / float(fs)
        ax.set_xlabel("Time (s)")

    # all traces
    for k in range(stack.shape[0]):
        ax.plot(x, stack[k], color="gray", alpha=0.25, linewidth=1)

    # mean
    m = np.nanmean(stack, axis=0)
    ax.plot(x, m, color="black", linewidth=2)

    ax.set_title(title)
    return stack


In [30]:


def to_1d_finite(x):
    arr = np.asarray(x, dtype=float).ravel()
    return arr[np.isfinite(arr)]

def safe_r(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if x.size < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def permute_pairing_null(x, y, n_perm=10000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if x.size < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan, np.array([])

    r_obs = np.corrcoef(x, y)[0, 1]
    r_null = np.empty(n_perm, float)
    for k in range(n_perm):
        y_sh = rng.permutation(y)
        r_null[k] = np.corrcoef(x, y_sh)[0, 1]
    p = np.mean(np.abs(r_null) >= np.abs(r_obs))
    return float(r_obs), float(p), r_null


def sample_random_indices(vol_len, n, window_s, Vsr, seed=0, exclude_idx=None, exclude_radius_s=0.5):
    """Uniform random timepoints, avoids edges + i<10, optional exclusion around true changepoints."""
    rng = np.random.default_rng(seed)
    margin = int(np.ceil(window_s * Vsr))
    min_i = max(margin, 10)                # because you index motor[i-10]
    max_i = vol_len - margin - 1
    if max_i <= min_i:
        return np.array([], dtype=int)

    valid = np.arange(min_i, max_i + 1)

    if exclude_idx is not None and len(exclude_idx) > 0:
        rad = int(np.ceil(exclude_radius_s * Vsr))
        bad = np.zeros(vol_len, dtype=bool)
        for ci in np.asarray(exclude_idx, dtype=int):
            lo = max(0, ci - rad); hi = min(vol_len, ci + rad + 1)
            bad[lo:hi] = True
        valid = valid[~bad[valid]]

    if valid.size == 0:
        return np.array([], dtype=int)
    replace = valid.size < n
    return rng.choice(valid, size=n, replace=replace).astype(int)

def temporal_null_pvalue_and_null(records, r_obs_mr, r_obs_rm, n_perm=2000, seed=0,
                                  window_s=3, fr_bin_s=0.0333, exclude_radius_s=0.5):
    """
    Temporal null: replace true changepoints with random timepoints (matched count per recording),
    then recompute r for MR and RM per permutation.

    Returns:
        p_time_mr, p_time_rm, rnull_mr, rnull_rm
    """
    rng = np.random.default_rng(seed)
    rnull_mr = np.empty(n_perm, float)
    rnull_rm = np.empty(n_perm, float)

    for k in range(n_perm):
        dca_mr_all, dfr_mr_all = [], []
        dca_rm_all, dfr_rm_all = [], []

        for rec in records:
            vol = rec["vol"]; cal = rec["cal"]
            volax = rec["volax"]; calax = rec["calax"]
            motor = rec["motor"]; spike = rec["spike"]
            changeP = rec["changeP"]
            Vsr = rec.get("Vsr", 500)
            Csr = rec.get("Csr", 30)

            rand_cp = sample_random_indices(
                vol_len=len(vol),
                n=len(changeP),
                window_s=window_s,
                Vsr=Vsr,
                seed=int(rng.integers(1_000_000_000)),
                exclude_idx=changeP,
                exclude_radius_s=exclude_radius_s
            )

            if rand_cp.size == 0:
                continue

            res0 = BS_change_diff(
                cal, vol, calax, volax,
                rand_cp.tolist(), motor, spike,
                window=window_s, Vsr=Vsr, Csr=Csr, fr_bin_s=fr_bin_s
            )

            dca_mr_all.extend(res0["motorToRest"]["CalMean"])
            dfr_mr_all.extend([v[0] for v in res0["motorToRest"]["FRMean"]])  # stored as [val]

            dca_rm_all.extend(res0["restToMotor"]["CalMean"])
            dfr_rm_all.extend([v[0] for v in res0["restToMotor"]["FRMean"]])

        rnull_mr[k] = safe_r(to_1d_finite(dfr_mr_all), to_1d_finite(dca_mr_all))
        rnull_rm[k] = safe_r(to_1d_finite(dfr_rm_all), to_1d_finite(dca_rm_all))

    # finite only for p-value
    rnull_mr_f = rnull_mr[np.isfinite(rnull_mr)]
    rnull_rm_f = rnull_rm[np.isfinite(rnull_rm)]

    p_time_mr = np.mean(np.abs(rnull_mr_f) >= np.abs(r_obs_mr)) if rnull_mr_f.size else np.nan
    p_time_rm = np.mean(np.abs(rnull_rm_f) >= np.abs(r_obs_rm)) if rnull_rm_f.size else np.nan

    return float(p_time_mr), float(p_time_rm), rnull_mr, rnull_rm



In [31]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrB.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
motCell = pathPyr[bsPyr == 'motor']
AllCalSR = list(r['CALsr'])

In [32]:
MotToRest =[]
RestToMot = []
MeanMotToRest = []
MeanRestToMot = []  
CalMotToRest = []
CalRestToMot = []
VolMotToRest = []
VolRestToMot = []
FrMotToRest = []
FrRestToMot = []
FrDiffMotToRest = []
FrDiffRestToMot = []
calSR = 30
volSR = 500
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    #print(l)
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    
    motor = motor[VolMask].to_numpy(dtype=int)   # ✅ instead of leaving it a Series

    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    if os.path.exists(fpath):
        pathSpike = fpath
        spikeId = pd.read_csv(pathSpike)
        spikeId = np.array(spikeId)
        spikeId = spikeId.flatten()
        spikeId = spikeId.tolist()
    else:
        pathSpike = os.path.join(l,r'SpikeIdx.csv')
        print('LOL')
        IspikeId = pd.read_csv(pathSpike)
        IspikeId = np.array(IspikeId)
        IspikeId = IspikeId.flatten()
        IspikeId = IspikeId.tolist()
        VolMask = VolMask.astype(bool)

        # 2. Create a mapping from Old Index -> New Index
        # This creates an array where every index contains the count of "True" frames before it
        new_mapping = np.cumsum(VolMask) - 1 

        # 3. Filter and Map
        # We keep the spike IF the mask was True at that spot...
        # ...AND we convert it to the new index using our mapping.
        spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        
        result = BS_change_diff(TraceC, TraceV, CalAX, VolAX, changeP,motor,spikeId)
        MotToRest.append(result['motorToRest'])
        RestToMot.append(result['restToMotor'])
        CalMotToRest.append(result['motorToRest']['Cal'])
        CalRestToMot.append(result['restToMotor']['Cal'])
        VolMotToRest.append(result['motorToRest']['Vol'])
        VolRestToMot.append(result['restToMotor']['Vol'])
        MeanMotToRest.extend(result['motorToRest']['CalMean'])
        MeanRestToMot.extend(result['restToMotor']['CalMean'])
        FrMotToRest.append(result['motorToRest']['FR'])
        FrRestToMot.append(result['restToMotor']['FR'])
        FrDiffMotToRest.extend(result['motorToRest']['FRMean'])
        FrDiffRestToMot.extend(result['restToMotor']['FRMean'])

    else:
        continue        



Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov2\cell1
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\07-08-2025\fov1\cell1
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov1\cell0
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov4\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov5\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov6\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\18-08-2025-ans\fov6\cell2
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\l\20-08-25-ans\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\17-09-2025-rugc42-wh-s1-ans\fov1\2\cell0
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\17-09-2025-rugc42-wh-s1-ans\fov1\2\cell1
LOL
Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\17-09-2025-rugc42-wh-

In [ ]:
MotToRest =[]
RestToMot = []
MeanMotToRest = []
MeanRestToMot = []  
CalMotToRest = []
CalRestToMot = []
VolMotToRest = []
VolRestToMot = []
FrMotToRest = []
FrRestToMot = []
FrDiffMotToRest = []
FrDiffRestToMot = []
calSR = 30
volSR = 500
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    #print(l)
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    
    motor = motor[VolMask].to_numpy(dtype=int)   # ✅ instead of leaving it a Series

    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    if os.path.exists(fpath):
        pathSpike = fpath
        spikeId = pd.read_csv(pathSpike)
        spikeId = np.array(spikeId)
        spikeId = spikeId.flatten()
        spikeId = spikeId.tolist()
    else:
        pathSpike = os.path.join(l,r'SpikeIdx.csv')
        print('LOL')
        IspikeId = pd.read_csv(pathSpike)
        IspikeId = np.array(IspikeId)
        IspikeId = IspikeId.flatten()
        IspikeId = IspikeId.tolist()
        VolMask = VolMask.astype(bool)

        # 2. Create a mapping from Old Index -> New Index
        # This creates an array where every index contains the count of "True" frames before it
        new_mapping = np.cumsum(VolMask) - 1 

        # 3. Filter and Map
        # We keep the spike IF the mask was True at that spot...
        # ...AND we convert it to the new index using our mapping.
        spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        
        result = BS_change_diff(TraceC, TraceV, CalAX, VolAX, changeP,motor,spikeId)
        MotToRest.append(result['motorToRest'])
        RestToMot.append(result['restToMotor'])
        CalMotToRest.append(result['motorToRest']['Cal'])
        CalRestToMot.append(result['restToMotor']['Cal'])
        VolMotToRest.append(result['motorToRest']['Vol'])
        VolRestToMot.append(result['restToMotor']['Vol'])
        MeanMotToRest.extend(result['motorToRest']['CalMean'])
        MeanRestToMot.extend(result['restToMotor']['CalMean'])
        FrMotToRest.append(result['motorToRest']['FR'])
        FrRestToMot.append(result['restToMotor']['FR'])
        FrDiffMotToRest.extend(result['motorToRest']['FRMean'])
        FrDiffRestToMot.extend(result['restToMotor']['FRMean'])

    else:
        continue        



In [33]:
import numpy as np
import os
import plotly.graph_objects as go

def flatten_numeric(x):
    """Flatten nested lists/arrays into 1D float array, remove non-finite."""
    out = []
    for item in x:
        if item is None:
            continue
        if isinstance(item, (list, tuple, np.ndarray)):
            out.extend(np.asarray(item).ravel().tolist())
        else:
            out.append(item)
    out = np.asarray(out, dtype=float)
    return out[np.isfinite(out)]
def save_plotly_svg_html(fig, svg_path, html_path):
    os.makedirs(os.path.dirname(svg_path), exist_ok=True)
    # HTML
    fig.write_html(html_path, include_plotlyjs="cdn")
    # SVG (requires kaleido)
    fig.write_image(svg_path, format="svg")
# flatten and clean
# mot = flatten_numeric(MeanMotToRest)
# res = flatten_numeric(MeanRestToMot)
mot = MeanMotToRest
res = MeanRestToMot
fig = go.Figure()

fig.add_trace(go.Box(
    y=mot,
    name="Motor→Rest",
    boxpoints="outliers"  # or "all" if you want all points
))

fig.add_trace(go.Box(
    y=res,
    name="Rest→Motor",
    boxpoints="outliers"
))

fig.update_layout(
    title="Mean Ca change around transitions",
    yaxis_title="Δ mean Ca (after − before)",
    template="plotly_white"
)

out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"  # change if you want
svg_path  = os.path.join(out_dir, "CaMeanChange_boxplot_pyr.svg")
html_path = os.path.join(out_dir, "CaMeanChange_boxplot_pyr.html")

save_plotly_svg_html(fig, svg_path, html_path)

fig = go.Figure()

# jittered x positions
x_mot = np.random.normal(loc=0, scale=0.04, size=len(mot))
x_res = np.random.normal(loc=1, scale=0.04, size=len(res))

fig.add_trace(go.Scatter(
    x=x_mot,
    y=mot,
    mode="markers",
    name="Motor→Rest",
    marker=dict(size=6, opacity=0.6),
))

fig.add_trace(go.Scatter(
    x=x_res,
    y=res,
    mode="markers",
    name="Rest→Motor",
    marker=dict(size=6, opacity=0.6),
))

fig.update_layout(
    title="Mean Ca change around transitions",
    yaxis_title="Δ mean Ca (after − before)",
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=["Motor→Rest", "Rest→Motor"],
        showgrid=False
    ),
    template="plotly_white"
)


out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
svg_path  = os.path.join(out_dir, "CaMeanChange_points_pyr.svg")
html_path = os.path.join(out_dir, "CaMeanChange_points_pyr.html")

save_plotly_svg_html(fig, svg_path, html_path)




In [34]:
import numpy as np
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def to_1d_finite(x):
    """Convert list/array (possibly nested) to 1D float array, keep finite only."""
    arr = np.asarray(x, dtype=float).ravel()
    return arr[np.isfinite(arr)]

def corr_and_fit(x, y):
    """
    Returns:
      r, p, slope, intercept
    Uses Pearson r; p via scipy if available, else p=np.nan.
    Fit uses np.polyfit (least squares).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]

    if x.size < 3 or np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan, np.nan, np.nan, np.nan

    # Pearson r
    r = np.corrcoef(x, y)[0, 1]

    # p-value (optional)
    try:
        from scipy.stats import pearsonr
        r2, p = pearsonr(x, y)
        r = r2
    except Exception:
        p = np.nan

    # linear fit y = slope*x + intercept
    slope, intercept = np.polyfit(x, y, 1)

    return float(r), float(p), float(slope), float(intercept)

def fit_line_xy(x, slope, intercept, n=100):
    """Generate line coords spanning data range."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0 or not np.isfinite(slope) or not np.isfinite(intercept):
        return np.array([]), np.array([])
    xs = np.linspace(np.min(x), np.max(x), n)
    ys = slope * xs + intercept
    return xs, ys


# ---- data (already filled by your loop) ----
dca_mr = to_1d_finite(MeanMotToRest)        # ΔCa Motor→Rest
dfr_mr = to_1d_finite(FrDiffMotToRest)      # ΔFR Motor→Rest

dca_rm = to_1d_finite(MeanRestToMot)        # ΔCa Rest→Motor
dfr_rm = to_1d_finite(FrDiffRestToMot)      # ΔFR Rest→Motor

# ensure equal lengths (safety)
n_mr = min(len(dca_mr), len(dfr_mr))
n_rm = min(len(dca_rm), len(dfr_rm))
dca_mr, dfr_mr = dca_mr[:n_mr], dfr_mr[:n_mr]
dca_rm, dfr_rm = dca_rm[:n_rm], dfr_rm[:n_rm]

# ---- stats + fits ----
r_mr, p_mr, a_mr, b_mr = corr_and_fit(dfr_mr, dca_mr)
r_rm, p_rm, a_rm, b_rm = corr_and_fit(dfr_rm, dca_rm)
r_obs_mr, p_pair_mr, rnull_pair_mr = permute_pairing_null(dfr_mr, dca_mr, n_perm=10000, seed=0)
r_obs_rm, p_pair_rm, rnull_pair_rm = permute_pairing_null(dfr_rm, dca_rm, n_perm=10000, seed=1)


# Null B: temporal null (random timepoints; requires `records` saved during loop)
if "records" in globals() and len(records) > 0:
    p_time_mr, p_time_rm, rnull_time_mr, rnull_time_rm = temporal_null_pvalue_and_null(
        records=records,
        r_obs_mr=r_mr,
        r_obs_rm=r_rm,
        n_perm=2000,
        seed=123,
        window_s=3,
        fr_bin_s=0.0333,
        exclude_radius_s=0.5
    )
else:
    p_time_mr, p_time_rm = np.nan, np.nan
    rnull_time_mr, rnull_time_rm = np.array([]), np.array([])


rnull_pair_mr = finite(rnull_pair_mr)
rnull_pair_rm = finite(rnull_pair_rm)
alpha = 0.05

crit_mr = np.quantile(np.abs(rnull_pair_mr), 1 - alpha)
crit_rm = np.quantile(np.abs(rnull_pair_rm), 1 - alpha)


fig_perm = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Pairing perm null (MR) | r_obs={r_obs_mr:.3f}, p={p_pair_mr:.2e}",
        f"Pairing perm null (RM) | r_obs={r_obs_rm:.3f}, p={p_pair_rm:.2e}"
    )
)

# MR histogram
fig_perm.add_trace(
    go.Histogram(
        x=rnull_pair_mr,
        nbinsx=60,
        name="Null r (MR)",
        opacity=0.75,
        hovertemplate="r=%{x:.3f}<br>count=%{y}<extra></extra>"
    ),
    row=1, col=1
)
fig_perm.add_vline(x=r_obs_mr, row=1, col=1, line_width=3)

# RM histogram
fig_perm.add_trace(
    go.Histogram(
        x=rnull_pair_rm,
        nbinsx=60,
        name="Null r (RM)",
        opacity=0.75,
        hovertemplate="r=%{x:.3f}<br>count=%{y}<extra></extra>"
    ),
    row=1, col=2
)
fig_perm.add_vline(x=r_obs_rm, row=1, col=2, line_width=3)
# MR critical region lines
fig_perm.add_vline(x=crit_mr, row=1, col=1, line_dash="dash")
fig_perm.add_vline(x=-crit_mr, row=1, col=1, line_dash="dash")

# RM critical region lines
fig_perm.add_vline(x=crit_rm, row=1, col=2, line_dash="dash")
fig_perm.add_vline(x=-crit_rm, row=1, col=2, line_dash="dash")


fig_perm.update_xaxes(title_text="Permuted correlation r", row=1, col=1)
fig_perm.update_xaxes(title_text="Permuted correlation r", row=1, col=2)
fig_perm.update_yaxes(title_text="Count", row=1, col=1)

fig_perm.update_layout(
    title="Permutation test (pairing): null distribution of r with observed r",
    template="plotly_white",
    showlegend=False
)

fig_perm.show()
svg_path2  = os.path.join(out_dir, "PermutationNull_r_hist_MR_RM.svg")
html_path2 = os.path.join(out_dir, "PermutationNull_r_hist_MR_RM.html")
save_plotly_svg_html(fig_perm, svg_path2, html_path2)

rnull_time_mr = finite(rnull_time_mr)
rnull_time_rm = finite(rnull_time_rm)

alpha = 0.05
crit_time_mr = np.quantile(np.abs(rnull_time_mr), 1 - alpha) if rnull_time_mr.size else np.nan
crit_time_rm = np.quantile(np.abs(rnull_time_rm), 1 - alpha) if rnull_time_rm.size else np.nan

fig_time = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Temporal null (MR) | r_obs={r_mr:.3f}, p_time={p_time_mr:.2e}",
        f"Temporal null (RM) | r_obs={r_rm:.3f}, p_time={p_time_rm:.2e}",
    )
)

# MR
fig_time.add_trace(go.Histogram(x=rnull_time_mr, nbinsx=60, opacity=0.75,
                                hovertemplate="r=%{x:.3f}<br>count=%{y}<extra></extra>"),
                   row=1, col=1)
fig_time.add_vline(x=r_mr, row=1, col=1, line_width=3)
if np.isfinite(crit_time_mr):
    fig_time.add_vline(x=crit_time_mr,  row=1, col=1, line_dash="dash")
    fig_time.add_vline(x=-crit_time_mr, row=1, col=1, line_dash="dash")

# RM
fig_time.add_trace(go.Histogram(x=rnull_time_rm, nbinsx=60, opacity=0.75,
                                hovertemplate="r=%{x:.3f}<br>count=%{y}<extra></extra>"),
                   row=1, col=2)
fig_time.add_vline(x=r_rm, row=1, col=2, line_width=3)
if np.isfinite(crit_time_rm):
    fig_time.add_vline(x=crit_time_rm,  row=1, col=2, line_dash="dash")
    fig_time.add_vline(x=-crit_time_rm, row=1, col=2, line_dash="dash")

fig_time.update_xaxes(title_text="Null correlation r (random timepoints)", row=1, col=1)
fig_time.update_xaxes(title_text="Null correlation r (random timepoints)", row=1, col=2)
fig_time.update_yaxes(title_text="Count", row=1, col=1)
fig_time.update_layout(title="Temporal permutation: random-timepoint null distribution of r",
                       template="plotly_white", showlegend=False)

fig_time.show()
svg_path_t  = os.path.join(out_dir, "TemporalPermutationNull_r_hist_MR_RM.svg")
html_path_t = os.path.join(out_dir, "TemporalPermutationNull_r_hist_MR_RM.html")
save_plotly_svg_html(fig_time, svg_path_t, html_path_t)

xs_mr, ys_mr = fit_line_xy(dfr_mr, a_mr, b_mr)
xs_rm, ys_rm = fit_line_xy(dfr_rm, a_rm, b_rm)

def fmt_rp(r, p):
    if not np.isfinite(r):
        return "r=nan"
    if np.isfinite(p):
        return f"r={r:.2f}, p={p:.2e}"
    return f"r={r:.2f}"

title_mr = f"Motor→Rest: ΔCa vs ΔFR ({fmt_rp(r_mr, p_mr)})"
title_rm = f"Rest→Motor: ΔCa vs ΔFR ({fmt_rp(r_rm, p_rm)})"

# ---- figure ----
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(title_mr, title_rm),
    shared_yaxes=True
)

# Scatter: Motor→Rest
fig.add_trace(
    go.Scatter(
        x=dfr_mr, y=dca_mr,
        mode="markers",
        name="Motor→Rest",
        marker=dict(size=7, opacity=0.65),
        hovertemplate="ΔFR=%{x:.2f}<br>ΔCa=%{y:.4f}<extra></extra>"
    ),
    row=1, col=1
)

# Fit line: Motor→Rest
fig.add_trace(
    go.Scatter(
        x=xs_mr, y=ys_mr,
        mode="lines",
        name="Fit MR",
        hoverinfo="skip"
    ),
    row=1, col=1
)

# Scatter: Rest→Motor
fig.add_trace(
    go.Scatter(
        x=dfr_rm, y=dca_rm,
        mode="markers",
        name="Rest→Motor",
        marker=dict(size=7, opacity=0.65),
        hovertemplate="ΔFR=%{x:.2f}<br>ΔCa=%{y:.4f}<extra></extra>"
    ),
    row=1, col=2
)

# Fit line: Rest→Motor
fig.add_trace(
    go.Scatter(
        x=xs_rm, y=ys_rm,
        mode="lines",
        name="Fit RM",
        hoverinfo="skip"
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="ΔFR (after − before) [Hz]", row=1, col=1)
fig.update_xaxes(title_text="ΔFR (after − before) [Hz]", row=1, col=2)
fig.update_yaxes(title_text="ΔCa (after − before)", row=1, col=1)

fig.update_layout(
    title="Transition coupling: ΔCa as a function of ΔFR",
    template="plotly_white",
    showlegend=False
)

out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
svg_path  = os.path.join(out_dir, "DeltaCa_vs_DeltaFR_MR_RM_pyr.svg")
html_path = os.path.join(out_dir, "DeltaCa_vs_DeltaFR_MR_RM_pyr.html")
save_plotly_svg_html(fig, svg_path, html_path)

fig.show()


In [ ]:
import numpy as np
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def flatten_traces(x):
    """
    Flatten arbitrarily nested containers and return list of 1D float arrays.
    Keeps only numeric 1D arrays/lists (your Ca traces).
    """
    out = []

    def _rec(obj):
        if obj is None:
            return

        # numpy array
        if isinstance(obj, np.ndarray):
            if obj.ndim == 1:
                a = obj.astype(float, copy=False)
                if a.size and np.any(np.isfinite(a)):
                    out.append(a)
            else:
                # if it's 2D/3D, treat first dim as stack of traces
                for k in range(obj.shape[0]):
                    _rec(obj[k])
            return

        # list/tuple: recurse
        if isinstance(obj, (list, tuple)):
            for it in obj:
                _rec(it)
            return

        # scalar -> ignore (not a trace)
        return

    _rec(x)
    return out

def pad_to_matrix(traces, pad_value=np.nan):
    if len(traces) == 0:
        return np.zeros((0, 0), float)
    L = max(len(t) for t in traces)
    M = np.full((len(traces), L), pad_value, dtype=float)
    for i, t in enumerate(traces):
        n = len(t)
        M[i, :n] = t
    return M

def add_all_gray_mean_black(fig, traces_any, fs, row, col, title, ylab=None):
    traces = flatten_traces(traces_any)  # ✅ robust flatten

    if len(traces) == 0:
        fig.add_annotation(
            text="No traces",
            x=0.5, y=0.5,
            xref=f"x{(row-1)*2+col} domain",
            yref=f"y{(row-1)*2+col} domain",
            showarrow=False
        )
        return

    M = pad_to_matrix(traces, pad_value=np.nan)
    mean_trace = np.nanmean(M, axis=0)
    t = np.arange(M.shape[1]) / float(fs)

    # gray traces
    for i in range(M.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=t, y=M[i, :],
                mode="lines",
                line=dict(width=1),
                opacity=0.25,
                hoverinfo="skip",
                showlegend=False
            ),
            row=row, col=col
        )

    # mean trace (black)
    fig.add_trace(
        go.Scatter(
            x=t, y=mean_trace,
            mode="lines",
            line=dict(width=3),
            showlegend=False
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text="Time (s)", row=row, col=col)
    if ylab is not None:
        fig.update_yaxes(title_text=ylab, row=row, col=col)

def save_plotly_svg_html(fig, svg_path, html_path):
    os.makedirs(os.path.dirname(svg_path), exist_ok=True)
    fig.write_html(html_path, include_plotlyjs="cdn")
    fig.write_image(svg_path, format="svg")  # needs kaleido


# -----------------------------
# Build the 1x2 figure
# -----------------------------
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    subplot_titles=("Ca traces: Motor→Rest", "Ca traces: Rest→Motor")
)

add_all_gray_mean_black(fig, CalMotToRest, fs=calSR, row=1, col=1, title="", ylab="Ca (a.u.)")
add_all_gray_mean_black(fig, CalRestToMot, fs=calSR, row=1, col=2, title="", ylab=None)

fig.update_layout(
    template="plotly_white",
    height=420,
    width=1100,
    title="Ca traces aligned to transitions",
    showlegend=False
)
fig.update_xaxes(showgrid=True, gridwidth=1)
fig.update_yaxes(showgrid=True, gridwidth=1)

out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
svg_path  = os.path.join(out_dir, "Ca_traces_MotRest_vs_RestMot_pyr.svg")
html_path = os.path.join(out_dir, "Ca_traces_MotRest_vs_RestMot_pyr.html")
save_plotly_svg_html(fig, svg_path, html_path)

fig.show()


In [ ]:


def _stack_trim_to_minlen(traces_any):
    traces = flatten_traces(traces_any)
    if len(traces) == 0:
        return None, 0
    lengths = [len(t) for t in traces if t is not None and len(t) > 0]
    if len(lengths) == 0:
        return None, 0
    Lmin = int(min(lengths))
    if Lmin <= 1:
        return None, 0
    M = np.full((len(traces), Lmin), np.nan, float)
    for i, t in enumerate(traces):
        M[i, :] = np.asarray(t[:Lmin], dtype=float)
    return M, Lmin

def save_plotly_svg_html(fig, svg_path, html_path):
    os.makedirs(os.path.dirname(svg_path), exist_ok=True)
    fig.write_html(html_path, include_plotlyjs="cdn")
    fig.write_image(svg_path, format="svg")  # requires kaleido


# -----------------------------
# Build 1x2 figure:
# Left: Motor→Rest (Ca + FR, dual y)
# Right: Rest→Motor (Ca + FR, dual y)
# -----------------------------
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Motor→Rest: mean Ca + mean FR", "Rest→Motor: mean Ca + mean FR"),
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
)

# ---------- Motor→Rest ----------
stack_ca_mr, Lca_mr = _stack_trim_to_minlen(CalMotToRest)
stack_fr_mr, Lfr_mr = _stack_trim_to_minlen(FrMotToRest) if ("FrMotToRest" in globals()) else (None, 0)

if (stack_ca_mr is not None) and (stack_fr_mr is not None):
    L = min(Lca_mr, Lfr_mr)
    x = np.arange(L) / float(calSR)  # same timebase as before

    y_ca = np.nanmean(stack_ca_mr[:, :L], axis=0)
    y_fr = np.nanmean(stack_fr_mr[:, :L], axis=0)

    fig.add_trace(go.Scatter(x=x, y=y_ca, mode="lines", name="Ca (Motor→Rest)"), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=y_fr, mode="lines", name="FR (Motor→Rest)"), row=1, col=1, secondary_y=True)
else:
    fig.add_annotation(
        text="Missing Motor→Rest Ca/FR",
        x=0.5, y=0.5, showarrow=False,
        xref="x domain", yref="y domain"
    )

# ---------- Rest→Motor ----------
stack_ca_rm, Lca_rm = _stack_trim_to_minlen(CalRestToMot)
stack_fr_rm, Lfr_rm = _stack_trim_to_minlen(FrRestToMot) if ("FrRestToMot" in globals()) else (None, 0)

if (stack_ca_rm is not None) and (stack_fr_rm is not None):
    L = min(Lca_rm, Lfr_rm)
    x = np.arange(L) / float(calSR)

    y_ca = np.nanmean(stack_ca_rm[:, :L], axis=0)
    y_fr = np.nanmean(stack_fr_rm[:, :L], axis=0)

    fig.add_trace(go.Scatter(x=x, y=y_ca, mode="lines", name="Ca (Rest→Motor)"), row=1, col=2, secondary_y=False)
    fig.add_trace(go.Scatter(x=x, y=y_fr, mode="lines", name="FR (Rest→Motor)"), row=1, col=2, secondary_y=True)
else:
    fig.add_annotation(
        text="Missing Rest→Motor Ca/FR",
        x=0.5, y=0.5, showarrow=False,
        xref="x2 domain", yref="y2 domain"
    )

# axis labels
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.update_xaxes(title_text="Time (s)", row=1, col=2)

fig.update_yaxes(title_text="Ca (a.u.)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="FR (Hz)",   row=1, col=1, secondary_y=True)

fig.update_yaxes(title_text="Ca (a.u.)", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="FR (Hz)",   row=1, col=2, secondary_y=True)

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=450,
    title="Mean Ca + mean FR around transitions (dual y-axis per transition type)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)

# save
out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
svg_path  = os.path.join(out_dir, "Mean_Ca_FR_dualY_MR_RM_pyr.svg")
html_path = os.path.join(out_dir, "Mean_Ca_FR_dualY_MR_RM_pyr.html")
save_plotly_svg_html(fig, svg_path, html_path)

fig.show()
